# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and processing the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL of the Croissant schema for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Identifier: {metadata.identifier if hasattr(metadata, 'identifier') else ''}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}")
print(f"Description: {metadata.description}")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else ''}")

## 2. Data Overview
Review available record sets and their details (all referenced by their `@id`).

We inspect the Croissant schema to list all record sets present, along with their fields.

In [ ]:
# List record sets and their fields. Use @id for referencing.

def print_recordsets_fields(dataset):
    if not hasattr(dataset.metadata, 'recordSet') or not dataset.metadata.recordSet:
        print("No record sets found in the dataset schema.")
        return []
    records_sets_ids = []
    for rs in dataset.metadata.recordSet:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
        rs_name = rs['name'] if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', None)
        print(f"RecordSet: {rs_id} | Name: {rs_name}")
        if hasattr(rs, 'field'):
            for fld in rs.field:
                fld_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else getattr(fld, '@id', None)
                fld_name = fld['name'] if isinstance(fld, dict) and 'name' in fld else getattr(fld, 'name', None)
                print(f"  Field: {fld_id} | Name: {fld_name}")
        records_sets_ids.append(rs_id)
    return records_sets_ids

record_set_ids = print_recordsets_fields(dataset)

## 3. Data Extraction
Load data from available record sets into pandas DataFrames using their `@id`. You can refer to the listed `record_set_ids` above to select the appropriate `@id`.

In [ ]:
# Extract data from all available record sets and print their columns (fields by @id)

dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        records_iter = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(records_iter)
        dataframes[record_set_id] = df
        print(f"Columns for RecordSet '{record_set_id}':\n", df.columns.tolist())
        print(df.head(), "\n")
# If the dataset does not define record sets, you may need to explore other entries manually.

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll choose fields using their `@id` from above.

In [ ]:
# Example EDA: Filter and normalize numeric field if available
# For demonstration, we select the first available dataframe and use its first numeric column.

import numpy as np

selected_record_set = None
if dataframes:
    # Find first non-empty DataFrame
    for rs_id, df in dataframes.items():
        if not df.empty:
            selected_record_set = rs_id
            break

if selected_record_set:
    df = dataframes[selected_record_set]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # For example, first numeric field
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].quantile(0.5)  # e.g., median
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by first non-numeric field if available
        other_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        group_field = other_fields[0] if other_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print(f"No numeric fields found in record set '{selected_record_set}'.")
else:
    print("No suitable data for analysis.")

## 5. Visualization
Visualize distributions or relationships between numeric and categorical fields using matplotlib or seaborn. Here we plot the chosen numeric field's distribution, if found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in RecordSet '{selected_record_set}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    if group_field:
        # Boxplot grouped by the group field
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook we loaded the metadata and records from the FAIR^2 Croissant dataset:

- Inspected record set and field structure by `@id` using mlcroissant
- Loaded records into pandas DataFrame(s) for processing
- Performed basic exploration and filtering of numeric fields
- Visualized distribution and group-based variations

For further analysis, consult the Croissant schema or the dataset documentation to clarify field semantics and usage guidelines.